In [9]:
import pandas as pd
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import Ridge
from scipy.sparse import hstack

### 1. Загрузите данные об описаниях вакансий и соответствующих годовых зарплатах из файла salary-train.csv.

In [12]:
train = pd.read_csv('salary-train.csv')
test = pd.read_csv('salary-test-mini.csv')

### 2. Проведите предобработку: 
• Приведите тексты к нижнему регистру. 

• Замените все, кроме букв и цифр, на пробелы — это облегчит дальнейшее разделение текста на слова. Для такой замены в строке text подходит следующий вызов: re.sub(’[ ^ a−zA−Z0−9]’,’␣’,text.lower())

In [13]:
def preprocess_text(text_data):
    pattern = '[^a-zA-Z0-9]'
    cleaned = re.sub(pattern, ' ', str(text_data).lower())
    return cleaned 

train_data = train.copy()
test_data = test.copy()
train_data['FullDescription'] = train_data['FullDescription'].apply(preprocess_text)
test_data['FullDescription'] = test_data['FullDescription'].apply(preprocess_text)

• Примените TfidfVectorizer для преобразования текстов в векторы признаков. Оставьте только те слова, которые встречаются хотя бы в 5 объектах (параметр min_df у TfidfVectorizer).

In [17]:
text_vectorizer = TfidfVectorizer(min_df=5)
train_text_features = text_vectorizer.fit_transform(train_data['FullDescription'])
test_text_features = text_vectorizer.transform(test_data['FullDescription'])

• Замените пропуски в столбцах LocationNormalized и ContractTime на специальную строку ’nan’. Код для этого был приведен выше. 

• Примените DictVectorizer для получения one-hot-кодирования признаков LocationNormalized и ContractTime.

In [15]:
categorical_cols = ['LocationNormalized', 'ContractTime']

for column_name in categorical_cols:
    train_data[column_name] = train_data[column_name].fillna('nan')
    test_data[column_name] = test_data[column_name].fillna('nan')
feature_encoder = DictVectorizer(sparse=True)
train_cat_features = feature_encoder.fit_transform(
    train_data[categorical_cols].to_dict('records')
)
test_cat_features = feature_encoder.transform(
    test_data[categorical_cols].to_dict('records')
)

• Объедините все полученные признаки в одну матрицу "объекты-признаки". Обратите внимание, что матрицы для текстов и категориальных признаков являются разреженными. Для объединения их столбцов нужно воспользоваться функцией scipy.sparse.hstack.

In [18]:
X_train_combined = hstack([train_text_features, train_cat_features])
X_test_combined = hstack([test_text_features, test_cat_features])
target_variable = train_data['SalaryNormalized']

### 3. Обучите гребневую регрессию с параметром alpha=1. Целевая переменная записана в столбце SalaryNormalized.

In [19]:
regression_model = Ridge(alpha=1)
regression_model.fit(X_train_combined, target_variable)
predictions = regression_model.predict(X_test_combined)

### 4. Постройте прогнозы для двух примеров из файла salary-test-mini.csv. Значения полученных прогнозов являются ответом на задание. Укажите их через пробел.

In [21]:
result_text = f"{predictions[0]:.2f} {predictions[1]:.2f}"
print(f"Ответ: {result_text}")

with open('ans.txt', 'w') as output_file:
    output_file.write(result_text)

Ответ: 56572.77 37196.14
